<a href="https://colab.research.google.com/github/cubaseuser123/Gpt-2-from-Scratch/blob/main/gpt2_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer & Self-Attention Implementation
This notebook expands our consolidated Bigram Language Model into a complete Transformer by adding single & multi-head self-attention, feedforward layers, residual connections, and layer normalizations as built in Andrej Karpathy's tutorial.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 32
# ------------

torch.manual_seed(1337)

#we start with downloading the model first
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

#let's read inside and inspect this dataset
with open("input.txt", "r", encoding="utf-8") as f:
  text = f.read()

#now we separate out all of the unique characters in this dataset
characters = sorted(list(set(text)))
vocab_size = len(characters)

#now let's create tokens out of this, both encoding and decoding. We are doing character level split here, but later we can try out different techniques for tokenization as well.
stoi = {ch: i for i, ch in enumerate(characters)}
itos = {i : ch for i, ch in enumerate(characters)}
encode = lambda s: [stoi[c] for c in s] #we take a string, and output a list of integers.
decode = lambda l: "".join([itos[i] for i in l]) #we take a list of integers and output a string.

#time to tokenize this entire thing. we encode the entire vocabulary into a torch tensor over here
data = torch.tensor(encode(text), dtype=torch.long)

#let's just split the training and testing data right here now.
n = int(0.9 * len(data)) #first 90 percent
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
  #we will generate a small batch of data of inputs x and targets y
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x,y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

#now we have to start coding out a bigram model to run these batch tokens through and get our very first DL model.
class BigramLanguageModel(nn.Module):

  def __init__(self):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embed) #it will go to the lookup table and each token will directly read off logits for the next token
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, targets=None):
    B, T = idx.shape
    tok_emb = self.token_embedding_table(idx) #idx and targets are both (B, T) tensor of integers
    # Debug print: Check the value of T before accessing position_embedding_table
    # If T > block_size, this will cause an out-of-bounds error for position_embedding_table
    if T > self.position_embedding_table.num_embeddings:
        print(f"DEBUG: T ({T}) is greater than block_size ({self.position_embedding_table.num_embeddings})!")
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (B, T, C)
    x = tok_emb + pos_emb
    logits = self.lm_head(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B * T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      # crop idx to the last block_size tokens
      idx_cond = idx[:, -block_size:]
      #getting the predictions
      logits, loss = self(idx_cond)
      logits = logits[:, -1, :]
      #now we apply softmax
      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim=-1)
    return idx

model = BigramLanguageModel()
m = model.to(device)

#now we are creating a pytorch adam optimizer for this
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    optimizer.step()

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=500)[0].tolist()))